In [0]:
%run ./01_config

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
def log_audit(table_name: str, operation: str, rows_affected: int, layer: str = ""):

    spark.sql(f"""
        INSERT INTO {catalog}.{operations_schema}.audit_log
        (run_id, batch, layer, table_name, operation, rows_affected, event_timestamp) 
        values('{run_id}', '{batch_id}', '{layer}', '{table_name}', '{operation}', {rows_affected}, current_timestamp())
        """)

In [0]:
def log_checkpoint(stage: str, status: str, rows: int = 0):
    spark.sql(f"""
        create table if not exists {catalog}.{operations_schema}.pipeline_checkpoint
        (run_id STRING, stage STRING, batch_id INT, status STRING, rows INT, ts TIMESTAMP)
        """)
    spark.sql(f"""
        insert into {catalog}.{operations_schema}.pipeline_checkpoint
        (run_id, stage, batch_id, status, rows, ts)
        values('{run_id}','{stage}', {batch_id},'{status}' , {rows}, current_timestamp())
        """)